Ray 是一个开源的分布式计算框架，用于构建和运行分布式应用程序。它支持多种编程语言，但最常用的是 Python。
- Ray 提供了简单易用的 API，可以让你轻松地将代码扩展到多核和多台机器上

# 初始化 Ray
- 在使用 Ray 之前，你需要先初始化它。这可以通过调用 ray.init() 来完成。
- ray.init() 会启动 Ray 的运行时环境，并返回一个 Ray 运行时的上下文对象。


## ray.init() 的参数
ray.init() 有许多参数可以配置，以下是一些常用的参数：
- address: 指定 Ray 集群的地址。如果不指定，默认会启动一个本地 Ray 运行时。
- num_cpus: 指定本地机器上可用的 CPU 核心数。
- num_gpus: 指定本地机器上可用的 GPU 数量。
- object_store_memory: 指定 Ray 对象存储的内存大小。
- dashboard_port: 指定 Ray Dashboard 的端口号。

In [1]:
import ray

# 初始化 Ray
ray.init()

# 定义一个 Ray 任务
@ray.remote
def hello_world():
    return "Hello, World!"

# 调用任务并获取结果
#  是一个同步调用
result = ray.get(hello_world.remote())
print(result)

# 关闭 Ray
ray.shutdown()

2025-10-28 18:37:53,155	INFO worker.py:1951 -- Started a local Ray instance.


Hello, World!


## 连接到 Ray 集群
- 如果你有一个 Ray 集群，可以通过指定 address 参数来连接到集群：

In [2]:
import ray

# 连接到 Ray 集群
ray.init(address="auto")

# 定义一个 Ray 任务
@ray.remote
def hello_world():
    return "Hello, World!"

# 调用任务并获取结果
result = ray.get(hello_world.remote())
print(result)

# 关闭 Ray
ray.shutdown()

ConnectionError: Could not find any running Ray instance. Please specify the one to connect to by setting `--address` flag or `RAY_ADDRESS` environment variable.

## 使用 Ray Dashboard
- Ray 提供了一个交互式仪表板，可以帮助你监控和调试 Ray 应用程序。默认情况下，Ray Dashboard 在本地运行时会监听端口 8265。你可以通过以下命令访问它：
- http://localhost:8265
- 如果你在集群环境中运行 Ray，可以通过指定 dashboard_port 参数来配置端口号：




In [ ]:
ray.init(address="auto", dashboard_port=8266)

## untime_env 参数是 Ray 用于指定运行时环境的高级功能。
- 它允许你在运行 Ray 任务和 actor 时，动态地指定依赖项和环境配置。
- 这在处理复杂的依赖项或需要隔离运行环境的场景中非常有用。

### runtime_env 的主要用途
- 安装依赖项：指定需要安装的 Python 包。
- 隔离环境：为每个任务或 actor 创建独立的运行环境。
- 加载文件和目录：将本地文件或目录上传到运行环境中。
- 指定 Conda 或 Pip 环境：使用预定义的 Conda 或 Pip 环境。


### runtime_env 是一个字典，可以包含以下键：
- pip：指定需要安装的 Pip 包。
- conda：指定需要使用的 Conda 环境。
- env_vars：指定环境变量。
- working_dir：指定工作目录。
- excludes：指定需要排除的文件或目录。
- eager_install：是否在任务启动时立即安装依赖项。
- py_modules：指定需要加载的 Python 模块。

In [ ]:
# 使用pip安装依赖环境

import ray

# runtime_env 指定了需要安装的 Pip 包 requests 和 numpy。
# 初始化 Ray，并指定运行时环境
ray.init(runtime_env={"pip": ["requests", "numpy"]})

@ray.remote
def fetch_url(url):
    import requests
    response = requests.get(url)
    return response.status_code

# 调用任务并获取结果
result = ray.get(fetch_url.remote("https://www.example.com"))
print(result)

# 关闭 Ray
ray.shutdown()

In [ ]:
# 使用 Conda 环境

import ray

# 初始化 Ray，并指定运行时环境
# 指定了需要使用的 Conda 环境，并安装了 numpy 和 pandas。
ray.init(runtime_env={"conda": {"dependencies": ["numpy", "pandas"]}})

@ray.remote
def compute_sum(numbers):
    import numpy as np
    return np.sum(numbers)

# 调用任务并获取结果
numbers = [1, 2, 3, 4, 5]
result = ray.get(compute_sum.remote(numbers))
print(f"Sum of numbers: {result}")

# 关闭 Ray
ray.shutdown()

## 加载文件和目录
- 假设你有一个本地目录 my_module，包含一些 Python 模块，你可以将其上传到运行环境中：

In [ ]:
import ray

# 初始化 Ray，并指定运行时环境
ray.init(runtime_env={"working_dir": "./my_module"})

@ray.remote
def use_module():
    from my_module import my_function
    return my_function()

# 调用任务并获取结果
result = ray.get(use_module.remote())
print(result)

# 关闭 Ray
ray.shutdown()

## 指定环境变量
- runtime_env 指定了环境变量 MY_VAR，其值为 my_value

In [ ]:
import ray

# 初始化 Ray，并指定运行时环境
ray.init(runtime_env={"env_vars": {"MY_VAR": "my_value"}})

@ray.remote
def print_env_var():
    import os
    return os.getenv("MY_VAR")

# 调用任务并获取结果
result = ray.get(print_env_var.remote())
print(result)

# 关闭 Ray
ray.shutdown()

### 动态指定 runtime_env
你也可以在定义任务或 actor 时动态指定 runtime_env，而不需要在 ray.init() 中全局指定

In [ ]:
import ray

# 初始化 Ray
ray.init()

@ray.remote(runtime_env={"pip": ["requests"]})
def fetch_url(url):
    import requests
    response = requests.get(url)
    return response.status_code

# 调用任务并获取结果
result = ray.get(fetch_url.remote("https://www.example.com"))
print(result)

# 关闭 Ray
ray.shutdown()

### @ray.remote 是 Ray 框架中的一个装饰器，
- 用于将普通的 Python 函数或类转换为 Ray 的任务（tasks）或 actor。
- 这是 Ray 实现分布式计算的核心机制之一。
- 通过 @ray.remote，你可以轻松地将代码分发到多个 CPU 核心或多个机器上运行。

#### @ray.remote 装饰器接受多个参数，用于控制任务或 Actor 的行为。以下是一些常用的参数：
- num_cpus: 指定任务或 Actor 需要的 CPU 核心数。
- num_gpus: 指定任务或 Actor 需要的 GPU 数量。
- memory: 指定任务或 Actor 需要的内存大小。
- object_store_memory: 指定任务或 Actor 需要的对象存储内存大小。
- resources: 指定任务或 Actor 需要的自定义资源。
- max_restarts: 指定 Actor 的最大重启次数（仅适用于 Actor）。
- max_task_retries: 指定任务的最大重试次数（仅适用于任务）。
- checkpoint_interval: 指定 Actor 的检查点间隔（仅适用于 Actor）。
- checkpoint_at_end: 指定是否在 Actor 结束时创建检查点（仅适用于 Actor）。
- retry_exceptions: 指定是否在任务失败时重试（仅适用于任务）。
- max_concurrency: 指定 Actor 的最大并发数（仅适用于 Actor）





In [5]:
import ray

# 初始化 Ray
ray.init(num_cpus=4, num_gpus=2)

# 定义一个任务，指定需要 2 个 CPU 核心
@ray.remote(num_cpus=2)
def compute_sum(numbers):
    return sum(numbers)

# 调用任务
result_ref = compute_sum.remote([1, 2, 3, 4, 5])

# 获取任务结果
result = ray.get(result_ref)
print(result)  # 输出: 15

# 定义一个 Actor，指定需要 1 个 GPU
@ray.remote(num_gpus=1)
class GPUActor:
    def __init__(self):
        pass

    def compute(self, x):
        return x * 2

# 创建一个 Actor 实例
actor = GPUActor.remote()

# 调用 Actor 的方法
result_ref = actor.compute.remote(10)

# 获取方法调用的结果
result = ray.get(result_ref)
print(result)  # 输出: 20

# 关闭 Ray
ray.shutdown()

2025-10-28 19:22:14,665	INFO worker.py:1951 -- Started a local Ray instance.


15
20


### 任务（Tasks）
任务是 Ray 中的基本执行单元，它们是无状态的，并且每次调用都会在 Ray 的调度系统中独立运行。任务的返回值是 ObjectRef，这是一个指向任务结果的引用。你可以通过 ray.get() 来获取任务的结果。

In [4]:
import ray

# 初始化 Ray
ray.init()

# 定义一个任务
@ray.remote
def add(a, b):
    return a + b

# 调用任务
result_ref = add.remote(1, 2)

# 获取任务结果
result = ray.get(result_ref)
print(result)  # 输出: 3

# 关闭 Ray
ray.shutdown()

2025-10-28 19:19:32,838	INFO worker.py:1951 -- Started a local Ray instance.


3


### Actor
- Actor 是 Ray 中的另一种执行单元，它们是有状态的，并且可以跨多个任务调用。
- Actor 的方法调用是异步的，并且每个方法调用都会返回一个 ObjectRef。
- 你可以通过 ray.get() 来获取方法调用的结果。

In [ ]:
import ray

# 初始化 Ray
ray.init()

# 定义一个 Actor
@ray.remote
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return self.value

    def get_value(self):
        return self.value

# 创建一个 Actor 实例
counter = Counter.remote()

# 调用 Actor 的方法
increment_ref = counter.increment.remote()
get_value_ref = counter.get_value.remote()

# 获取方法调用的结果
increment_result = ray.get(increment_ref)
get_value_result = ray.get(get_value_ref)

print(increment_result)  # 输出: 1
print(get_value_result)  # 输出: 1

# 关闭 Ray
ray.shutdown()